# Lesson 17: Training the action expert

## The idea before the implementation

**Train action generation while preserving the backbone.** SmolVLA learns a velocity field that transforms noise into action chunks. During training, a demonstrated action chunk and a noise sample define a known path and velocity target. The expert learns from that target while reading the observation and instruction context.

**Where you work.** You work on one real training update. The course freezes the VLM backbone and updates permitted expert and projection parameters. The noisy input and target are constructed from demonstrations; the loss excludes padded times and unused action channels. Sampling later uses fixed weights and does not know the demonstrated target.

**Pause and explain.** Explain why setting a module to evaluation mode and freezing its parameter gradients solve different problems. Check both before you interpret an update.

Keep [the concept guide](../concepts/smolvla.md) open beside this notebook for the full system diagrams and lesson map.

This lesson uses small code cells to show the real SmolVLA path. Run cells in order. The learner notebook has exercises for you to complete.

**Result:** a flow-matching update that changes the action expert while it keeps the VLM frozen.

**Notebook pair:** [Reference answers](17-smolvla-training-reference.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
import json
import os
import random
import time
from collections.abc import Mapping

import numpy as np
import torch
import yaml
from lerobot.policies.smolvla import SmolVLAPolicy
from lerobot.processor import load_pretrained_policy_processors
from torch import nn
from torch.utils.data import default_collate

from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.interfaces import ActionProcessor, ObservationProcessor
from vla_lab.teaching.smolvla import (
    CallbackSmolVLAPolicy,
    SmolVLAAdapter,
    euler_sample,
    load_prepared_smolvla,
    select_trainable_parameters,
    train_mode_with_frozen_vlm,
)
from vla_lab.teaching.training import optimizer_update

## Model map: the loss updates an action expert

The VLM is the large vision-language backbone. The action expert is a smaller stack that reads the VLM context and predicts action velocity. In this course, the checkpoint weights for both are supplied. Your update changes the permitted expert and projection parameters; the VLM stays frozen. A frozen VLM has `requires_grad=False`, receives no optimizer update, and stays in evaluation mode so dropout does not add noise.

Training creates one noisy point on a straight path from a clean action chunk $a$ to Gaussian noise $\varepsilon$, and asks the model for the velocity along that path:

$$x_t=(1-t)\,a+t\,\varepsilon, \qquad u=\varepsilon-a$$

$$L=\frac{\sum_{b,j,d}v_{b,j}\left(\hat{v}_{b,j,d}-u_{b,j,d}\right)^2}{\max\!\left(D\sum_{b,j}v_{b,j},\ 1\right)}$$

Where:
* $a$ - the clean action chunk, shape $(B,T,32)$ after padding to the expert's width.
* $\varepsilon\sim\mathcal{N}(0,I)$ - Gaussian noise with the shape of $a$.
* $t\in[0,1]$ - the flow time, one value per example; $t=0$ is clean, $t=1$ is pure noise.
* $x_t$ - the point on the path at time $t$.
* $u$ - the target velocity; it is constant along the straight path.
* $\hat{v}=v_\theta(x_t,t,c)$ - the expert's predicted velocity, given the VLM context $c$.
* $b$, $j$, $d$ - indices of the example, action time step, and action value.
* $D$ - the number of real action values ($D=7$, `adapter.action_dim`); the sum over $d$ uses only those, not the 25 zero-padded columns.
* $v_{b,j}$ - 1 for a real time step, 0 when `action_is_pad[b, j]` is `True`; padded times do not enter the mean.

<svg xmlns="http://www.w3.org/2000/svg" width="780" height="138" viewBox="0 0 780 138" role="img" aria-label="Train the SmolVLA action path">
  <title>Train the SmolVLA action path</title>
  <desc>The frozen VLM supplies context. Only permitted expert and projection weights are updated.</desc>
  <rect width="780" height="138" rx="8" fill="#f8fafc"/>
  <text x="12" y="22" font-family="sans-serif" font-size="15" font-weight="bold" fill="#0f172a">Train the SmolVLA action path</text>
  <rect x="12" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="97.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Clean actions,</text>
  <text x="97.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">noise, and time</text>
  <path d="M 186 67 H 203 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="207" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="292.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Interpolate</text>
  <text x="292.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">a noisy chunk</text>
  <path d="M 381 67 H 398 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="402" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="487.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Predict velocity</text>
  <text x="487.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">with context</text>
  <path d="M 576 67 H 593 m -5 -4 l 5 4 -5 4" fill="none" stroke="#334155" stroke-width="2"/>
  <rect x="597" y="38" width="171" height="58" rx="6" fill="#dbeafe" stroke="#1d4ed8"/>
  <text x="682.5" y="61" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">Masked loss</text>
  <text x="682.5" y="78" text-anchor="middle" font-family="sans-serif" font-size="12" fill="#0f172a">and weight update</text>
  <text x="12" y="120" font-family="sans-serif" font-size="12" fill="#334155">The frozen VLM supplies context. Only permitted expert and projection weights are updated.</text>
</svg>

Images, text, and projected state provide prefix context. The known target velocity is noise minus demonstrated actions. Compare the expert prediction with this target on real action channels and valid times, then backpropagate and update the permitted weights.


PyTorch tensors store values; `requires_grad` asks PyTorch to record operations that can update a parameter. `loss.backward()` fills each trainable parameter's `.grad`. `optimizer.step()` changes parameters with gradients. Clearing gradients first prevents values from two updates from accumulating. For one real action value, take $a=2$, $\varepsilon=-1$, and $t=0.25$. Then $x_t=1.25$ and target $u=-3$. If the model predicts $-2$, the squared error is $1$. A padded action column does not enter the mean. A check on parameter values and gradients is stronger than checking a printed loss.

## Exercise 17.1 — Make one real flow update

**Purpose.** Write the loss callback that connects the supplied model to PyTorch's optimizer. This is the operation used during adaptation.

`batch` is the preprocessed `dict[str, Tensor]`: camera images, state, language token IDs and mask, `action` $(B,T,7)$, and `action_is_pad` $(B,T)$. First call `adapter.prepare_inputs(batch)`. It returns a `VLAInputs` dict (from `vla_lab.teaching.interfaces`) in the layout the SmolVLA model functions take:
* `images` - list of camera tensors, one per camera.
* `img_masks` - list of `(B,)` bool tensors, `True` where the camera image exists.
* `lang_tokens`, `lang_masks` - token IDs and their `True`-for-real mask, both $(B,L_t)$.
* `state` - the robot state, padded to the model's state width.
* `actions` - $(B,T,32)$: the 7 action values padded with zeros to 32 columns.

Draw `noise` with the shape of `inputs["actions"]` and `time` with shape $(B,)$. Then use the supplied helpers:
* `flow_training_pair(actions, noise, time) -> (x_t, u)` - the two path formulas above.
* `adapter.velocity(inputs, x_t, time) -> v` - the predicted velocity, $(B,T,32)$.
* `masked_velocity_loss(v, u, action_is_pad, adapter.action_dim)` - the loss $L$ above.

Return `(scalar_loss, {"flow_mse": scalar_loss})`. The callback signature is accepted by `optimizer_update`. Keep the VLM frozen before the optimizer is built. Do not detach the predicted velocity: gradients must reach the action expert.

More behaviour checks for this exercise: `check_learner_flow_loss` in [lesson17_checks.py](lesson17_checks.py).

In [ ]:
def learner_flow_loss(
    adapter: SmolVLAAdapter,
    batch: Mapping[str, torch.Tensor],
    noise: torch.Tensor | None = None,
    time: torch.Tensor | None = None,
) -> tuple[torch.Tensor, dict[str, torch.Tensor]]:
    # TODO: prepare inputs, make noisy actions and target, predict velocity,
    # and return the padding-aware scalar loss with a flow_mse metric.
    raise NotImplementedError("Connect the flow loss to the real adapter")


# The optimizer calls: optimizer_update(adapter, batch, optimizer, learner_flow_loss).
# Keep the callback above as the single loss path for training and validation.

The next untagged cells check gradient flow and VLM freezing on a deterministic toy network. The tagged experiment uses `optimizer_update` with the local published model and data. The source VLM stays frozen; this cell checks its parameters before and after.

In [ ]:
# Tiny adapter fixture exercises the exact learner callback on CPU.
class TinyAdapter:
    action_dim: int

    def __init__(self: "TinyAdapter") -> None:
        self.action_dim = 1
        self.scale = nn.Parameter(torch.tensor(0.0))

    def prepare_inputs(
        self: "TinyAdapter", batch: Mapping[str, torch.Tensor]
    ) -> dict[str, torch.Tensor]:
        return {"actions": batch["action"], "state": batch["state"]}

    def velocity(
        self: "TinyAdapter",
        inputs: Mapping[str, torch.Tensor],
        x_t: torch.Tensor,
        time: torch.Tensor,
    ) -> torch.Tensor:
        return self.scale.expand_as(x_t)


toy = TinyAdapter()
toy_batch = {
    "action": torch.tensor([[[2.0], [100.0]]]),
    "action_is_pad": torch.tensor([[False, True]]),
    "state": torch.zeros(1, 1),
}
toy_noise = torch.tensor([[[-1.0], [-100.0]]])
toy_time = torch.tensor([0.25])
toy_loss, toy_metrics = learner_flow_loss(toy, toy_batch, toy_noise, toy_time)
assert torch.allclose(toy_loss, torch.tensor(9.0))
toy_loss.backward()
assert torch.allclose(toy.scale.grad, torch.tensor(6.0))
assert torch.equal(toy_metrics["flow_mse"], toy_loss)
print("The callback masks the padded step and sends gradients to the expert.")

# More behaviour checks: lesson17_checks.py, next to this notebook.
from lesson17_checks import check_learner_flow_loss

check_learner_flow_loss(learner_flow_loss)

**Explain (2–3 sentences).** Which tensors get gradients? Why must the VLM remain in evaluation mode when its parameters are frozen?

In [ ]:
# Setup: fixed held-out inputs and a matched published-policy baseline.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 1706
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla("published", DATA, device=DEVICE)
FRAME_INTERVAL_SECONDS = 0.1  # Dataset labels are stored at 10 FPS.
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
TRAIN_DS = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "train",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
VALID_DS = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
VALID_IDS = torch.linspace(0, len(VALID_DS) - 1, 8).round().long().tolist()
VALID_RAW = default_collate([VALID_DS[index] for index in VALID_IDS])
TASKS = yaml.safe_load((CONFIGS / "data" / "episodes.yaml").read_text())
TASK_PROMPT = TASKS["tasks"]["bowl_plate"]["task_name"]
VALID_RAW["task"] = [TASK_PROMPT] * len(VALID_IDS)
VALID_BATCH = PREPROCESSOR(VALID_RAW)
VALID_BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in VALID_BATCH.items()
}
POLICY = POLICY.to(DEVICE).eval()
ADAPTER = SmolVLAAdapter(POLICY)
FIXED_NOISE = torch.randn_like(ADAPTER.prepare_inputs(VALID_BATCH)["actions"])
FIXED_TIME = torch.linspace(0.1, 0.9, len(VALID_IDS), device=DEVICE)
with torch.no_grad():
    BEFORE_LOSS, _ = learner_flow_loss(ADAPTER, VALID_BATCH, FIXED_NOISE, FIXED_TIME)
SPLITS = yaml.safe_load((CONFIGS / "eval" / "splits.yaml").read_text())
STATE_IDS = SPLITS["tasks"]["bowl_plate"]["development_state_ids"][:3]


def run_libero_set(
    policy: SmolVLAPolicy,
    preprocessor: ObservationProcessor,
    postprocessor: ActionProcessor,
    run_seed: int,
) -> list[dict[str, int | bool | str]]:
    rows = []
    for state_id in STATE_IDS:
        seed = run_seed + state_id
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
        environment = open_libero_task("bowl_plate", state_id, CONFIGS, DATA, control_hz=20)
        try:
            result = run_policy_episode(
                environment,
                policy,
                preprocessor,
                postprocessor,
                "bowl_plate",
                state_id,
                CONFIGS,
                partition="development",
                task_prompt=TASK_PROMPT,
                reset_seed=seed,
                control_hz=20,
            )
            rows.append(
                {
                    "state_id": state_id,
                    "seed": seed,
                    "success": result.success,
                    "steps": result.steps,
                    "termination_reason": result.termination_reason,
                    "control_hz": result.control_hz,
                }
            )
        finally:
            environment.close()
    return rows


BASELINE_POLICY = CallbackSmolVLAPolicy(POLICY, sampler=euler_sample, num_steps=8, seed=SEED)
BASELINE_EPISODES = run_libero_set(BASELINE_POLICY, PREPROCESSOR, POSTPROCESSOR, SEED)
print(
    {
        "held_out_loss_before": float(BEFORE_LOSS),
        "baseline_successes": sum(int(row["success"]) for row in BASELINE_EPISODES),
        "episodes": len(BASELINE_EPISODES),
        "state_ids": STATE_IDS,
    }
)

Both policy scores use three development states and fixed seeds. The simulator sends one relative action per 20 Hz control step. The dataset stores action labels at 10 FPS; it does not resample them.

In [ ]:
# Train: update the expert only, then save the policy and processors.
train_mode_with_frozen_vlm(POLICY)
TRAINABLE = select_trainable_parameters(POLICY)
OPTIMIZER = torch.optim.AdamW(TRAINABLE, lr=1e-5, weight_decay=1e-4, foreach=False)
VLM_BEFORE = [
    parameter.detach().cpu().clone() for parameter in POLICY.model.vlm_with_expert.vlm.parameters()
]
TRAIN_GENERATOR = torch.Generator().manual_seed(SEED)
MAX_SECONDS = 30 * 60
MAX_UPDATES = 300
BATCH_SIZE = 1
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    MEMORY_LIMIT_BYTES = min(
        48 * 1024**3, int(torch.cuda.get_device_properties(0).total_memory * 0.85)
    )
else:
    MEMORY_LIMIT_BYTES = 48 * 1024**3
STARTED = time.monotonic()
LOSSES = []
for _step in range(MAX_UPDATES):
    if time.monotonic() - STARTED >= MAX_SECONDS:
        break
    indices = torch.randint(len(TRAIN_DS), (BATCH_SIZE,), generator=TRAIN_GENERATOR).tolist()
    RAW_BATCH = default_collate([TRAIN_DS[index] for index in indices])
    RAW_BATCH["task"] = [TASK_PROMPT] * BATCH_SIZE
    BATCH = PREPROCESSOR(RAW_BATCH)
    BATCH = {
        key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
        for key, value in BATCH.items()
    }
    METRICS = optimizer_update(ADAPTER, BATCH, OPTIMIZER, learner_flow_loss)
    LOSSES.append(float(METRICS["loss"]))
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        if torch.cuda.max_memory_allocated() > MEMORY_LIMIT_BYTES:
            raise MemoryError("GPU use exceeded the 48 GB maximum or the local device budget")
assert len(LOSSES) >= 100, (
    "Use at least 100 updates before you describe this as an adaptation result."
)
ADAPTER.eval()
with torch.no_grad():
    AFTER_LOSS, _ = learner_flow_loss(ADAPTER, VALID_BATCH, FIXED_NOISE, FIXED_TIME)
assert all(
    torch.equal(before, after.detach().cpu())
    for before, after in zip(VLM_BEFORE, POLICY.model.vlm_with_expert.vlm.parameters(), strict=True)
)
ELAPSED_SECONDS = time.monotonic() - STARTED
PEAK_GPU_GB = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None
OUTPUT = (
    WORKSPACE
    / ".local-data/course-results"
    / ("smolvla-training-17-" + time.strftime("%Y%m%d-%H%M%S") + "-" + str(os.getpid()))
)
OUTPUT.mkdir(parents=True, exist_ok=False)
POLICY.save_pretrained(OUTPUT)
PREPROCESSOR.save_pretrained(OUTPUT)
POSTPROCESSOR.save_pretrained(OUTPUT)
print(
    {
        "updates": len(LOSSES),
        "seconds": round(ELAPSED_SECONDS, 1),
        "peak_gpu_gb": PEAK_GPU_GB,
        "held_out_loss_after": float(AFTER_LOSS),
        "checkpoint": str(OUTPUT),
    }
)

In [ ]:
# Reload: check fixed-noise parity and evaluate the saved policy on the same states.
RELOADED_POLICY = SmolVLAPolicy.from_pretrained(OUTPUT, config=POLICY.config).to(DEVICE).eval()
RELOADED_PRE, RELOADED_POST = load_pretrained_policy_processors(
    str(OUTPUT),
    preprocessor_overrides={
        "tokenizer_processor": {
            "tokenizer_name": str(DATA / "models/smolvlm2-500m-video-instruct")
        },
        "device_processor": {"device": DEVICE},
    },
)
RELOADED_BATCH = RELOADED_PRE(VALID_RAW)
RELOADED_BATCH = {
    key: value.to(DEVICE) if isinstance(value, torch.Tensor) else value
    for key, value in RELOADED_BATCH.items()
}
RELOADED_ADAPTER = SmolVLAAdapter(RELOADED_POLICY)
RELOADED_ADAPTER.eval()
with torch.no_grad():
    RELOADED_INPUTS = RELOADED_ADAPTER.prepare_inputs(RELOADED_BATCH)
    RELOADED_LOSS, _ = learner_flow_loss(RELOADED_ADAPTER, RELOADED_BATCH, FIXED_NOISE, FIXED_TIME)
    SAVED_SAMPLE = ADAPTER.sample_chunk(
        ADAPTER.prepare_inputs(VALID_BATCH), noise=FIXED_NOISE, num_steps=8
    )
    RELOADED_SAMPLE = RELOADED_ADAPTER.sample_chunk(RELOADED_INPUTS, noise=FIXED_NOISE, num_steps=8)
assert torch.allclose(SAVED_SAMPLE, RELOADED_SAMPLE, atol=1e-5)
assert torch.allclose(AFTER_LOSS, RELOADED_LOSS, atol=1e-6)
RELOADED_CALLBACK = CallbackSmolVLAPolicy(
    RELOADED_POLICY, sampler=euler_sample, num_steps=8, seed=SEED
)
AFTER_EPISODES = run_libero_set(RELOADED_CALLBACK, RELOADED_PRE, RELOADED_POST, SEED)
REPORT = {
    "task": "bowl_plate",
    "task_prompt": TASK_PROMPT,
    "dataset_revision": TASKS["dataset_revision"],
    "train_episode_ids": TASKS["tasks"]["bowl_plate"]["train_episode_ids"],
    "validation_episode_ids": TASKS["tasks"]["bowl_plate"]["validation_episode_ids"],
    "frame_interval_seconds": FRAME_INTERVAL_SECONDS,
    "dataset_label_rate_hz": 10,
    "simulator_control_hz": 20,
    "action_chunk_size": ACTION_CHUNK_SIZE,
    "seed": SEED,
    "held_out_indices": VALID_IDS,
    "state_ids": STATE_IDS,
    "training_config": {
        "learning_rate": 1e-5,
        "weight_decay": 1e-4,
        "batch_size": BATCH_SIZE,
        "max_updates": MAX_UPDATES,
        "max_seconds": MAX_SECONDS,
    },
    "updates": len(LOSSES),
    "elapsed_seconds": ELAPSED_SECONDS,
    "peak_gpu_gb": PEAK_GPU_GB,
    "held_out_loss_before": float(BEFORE_LOSS),
    "held_out_loss_after": float(RELOADED_LOSS),
    "baseline_episodes": BASELINE_EPISODES,
    "adapted_episodes": AFTER_EPISODES,
    "checkpoint": str(OUTPUT),
    "fixed_noise_reload_parity": True,
}
(OUTPUT / "course_report.json").write_text(json.dumps(REPORT, indent=2, sort_keys=True) + "\n")
print(
    {
        "baseline_successes": sum(int(row["success"]) for row in BASELINE_EPISODES),
        "adapted_successes": sum(int(row["success"]) for row in AFTER_EPISODES),
        "episodes_per_policy": len(STATE_IDS),
        "report": str(OUTPUT / "course_report.json"),
    }
)